# 🕵️ Dia 57 — Super-linhas do tempo com Plaso (log2timeline, pinfo, psort)

**Investigador Cyber 90 Dias** · Fase 4 · Rank Jutsu `B`

> Baseado no *Day 57: Super-timelines with Plaso (log2timeline, pinfo, psort)* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-4-digital-forensics-incident-investigation/day-57.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

O pacote do notebook `SA-CX-07` (10.20.40.57), da caixa da agência Centro da Cooperativa Serra Azul, chegou: proxy, firewall, Security, MFT e webmail. Você jogou tudo no Plaso e saiu um arquivo enorme. Rafa pergunta: *"Quantos eventos do firewall tem aí dentro? E em que ano o proxy diz que a caixa clicou?"*

Neste notebook você vai:

1. **Carregar o EVID-001 do LAB-P4** (com plano B offline) e registrar o SHA-256 de cada arquivo.
2. **Rodar um mini-Plaso em Python puro**: ver o firewall e o portal sumirem sem erro.
3. **Reproduzir a inferência de ano do syslog** e o erro que ela causa.
4. **Gerar a tabela de cobertura, a janela 02:45–04:00 UTC (l2tcsv) e a fatia de ±5 min.**
5. **Repetir no pacote do `SA-CX-07`** e jogar o quiz da cobertura.

🐍 Só biblioteca padrão. O Plaso de verdade roda via Docker (célula opcional no fim); aqui o objetivo é entender o que ele faz por dentro.

## 1. Carregando as evidências (rede primeiro, cópia embutida depois)

In [ ]:
# ------------------------------------------------------------
# CARREGADOR OFFLINE-FIRST
# Tenta baixar cada arquivo do GitHub (raw). Se a rede falhar,
# usa a cópia EMBUTIDA aqui embaixo (comprimida para caber no notebook).
# Nos dois casos, calculamos o SHA-256: é a nossa "cadeia de custódia" de bolso.
# ------------------------------------------------------------
import base64          # para decodificar as cópias embutidas
import hashlib         # SHA-256 de cada arquivo carregado
import urllib.request  # download simples, sem bibliotecas externas
import zlib            # as cópias embutidas estão comprimidas com zlib

# Endereço "cru" dos dados do LAB-P4 no repositório original (ThiefDroppers)
RAW_LAB = ("https://raw.githubusercontent.com/thiefdroppers/cyber-investigator-skills/main/"
           "curriculum/phase-4-digital-forensics-incident-investigation/resources/case-lab-p4/")
# Endereço "cru" dos dados sintéticos da Operação Isca neste repositório
RAW_BR = ("https://raw.githubusercontent.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/main/"
          "caso-continuo/dados/")

# Plano B: cópias fiéis dos arquivos, feitas a partir do repositório local.
EMBUTIDOS = {
    "lab:logs/bastion01-auth.log": """
eNq1m01v20YQhu/5FTzGhxo7s/OxS6AHA23RS3rppUBQBPpyIkSWVFFK4n9fWi7S2HztOuQ0EBLHMPRwd/nuzD603swODeWGpE3c
EjXzWXdc77aJmq77sHzLVJL/2TZXi8Vqf1wtm/1pvlkvPq5um+vdodntu8vZ8ma9ba4Pu5uG0mX/0v5vbva7w7FRUpG7d+K2+fkn
VqXa/P7rFau1f5Ub+uNL2szmqbvdHj+sjutF6t83Xa+371eH/WG9Paar/s+rNw8ukZ+6xP3s5t1pu/7y+u67bbfquv6HLtrmn6/6
a11t+wHcXfapW31z7a9P6+WPlBJdNPPb5vy/dPGVaq14myugkkyYGE6UoibmfImSnrrE/2NipOnXQqlNGVG1p/4yW2/upmXWdZ93
h/u3X28/zTbr5T3mm+nhlC/7uaF8KXo/P1I5lfP8POSRIp49wTvsdsdnCP1iDAksiOAvGdF8tvh42j8HNADMEFheAtwdZovN6hkg
CwAKvJnrE8Du0+Ldfw4spwGH24QGxuklAzuuuucWLjvAUUE4egnuND9tj6dngP2XQ2BGiWN+CfD6uL//92liRUS0cpwj0qYMeIo2
W5ZxadPhzZ/7HR0RNCRtJgDYRxAALSRtVgCQ4Zr5lLRZBRyBM1kC0uZoWIpaBa4hafPh+ARXmpxi0lYcEOGGkikibdUAL6MVzDwq
bbmfriFB4BzmiLRlIgBUCJSItGVQTBXXnKwT0paZAAduI9mmpy2DUqoto7Yne0TaMiim2vbfBMASkrYsGRAVVdNcA9KWZThCw5VH
0ri0SQEEQveiUEjaFA2J0aIJh6RNKwDCmiN5StoMzaTCmZSAtPmwR/A2oY1fNCRtbgDIcB4tJm2gmjo+dIhHpK1kwBO0gUkZlzZQ
rb1V1GdJDUlbHbb/BcsSTSFpqwUAGdUcpQlpE9AllDaj2195etqEEsDB46jmiLQJaEoq7iRVQtImVAERVgKNsCTCDHgMV3CcJRHQ
kFTcSWqIJZEsAAhLgIZYEsmP9uT+WJraBG/LKZZEJAEO9E0WYEnkcZt1xkGFYCGWRB73WWegoHvRYiyJ6HBGCXd2FmFJRAvgEWq7
bJwlESNAgA7BQiyJOJpCfJeEWBJxNIfwoGhTLImUYaoZt3YWYEmkEsAxHFaIJZFaATCjHsFjLIkmAUQogjzCkigNpzTj6u3jLIk+
fgJwJhAkhFgSJQdAWK49xJIoCwDCZ0U+xZIoK+Ao6vo9wJJoHg5L2gRxIZZEcwVAuPN7jCVRUE2lzTBtEZZE1QBP0H5SxlkSBbVN
+xYBEUIsiZoCIGzGS4glUSsACItAmWJJ1A1woD8uAZZE3QFO4DyGWBItDIBwNykxlkTLMN/WvxAxwpJozYDHKN1lnCXRWgABep8S
YkksMQDC9r+GWBJLCoCK9uQ6xZIYDYuN9y/ECbAkBnoExw+jaoglMVC7HT/drjGWxHJGRHToqBGWxMAJ33HnWsdZEgNn+9IfbBAh
xJIYaA9KS3DRQiyJSUVAuGZTLIkpAQ5qkWsKsCSmCnDowUZNIZbEFC0c0ls1xVgSA9YCW9CaIiyJaQU8giMcZ0nMBBDQQ9KaQiyJ
gU6ywoekNYVYEvMMgGjHqmmKJbEySBsl2JPXFGBJbNhm9TiCuBBLYtUQEOxalWIsiacMiBmljSIsiQ8dRs8TdGvSOEviNBwRtqyV
QiyJcwJAgsAQS+LsAMgQOMWS+FBb9Bzk7CoFWBIXBjiBN3+IJXEZjg//HmilGEviooBIMG0RlsSH3QlhzVt5nCVxzYAASw2HWBJX
NIVwA+EQS+LD/oAy7JUrT7Ekbglw0G+QVw6wJG4GcOjxXuUQS+IuAAh3E46xJF6G96bAk3Dlh58a+b6100TMiFSeIn3vZxz+vYb7
DzlQcvghB4C97Y6rm+UPm9379Xb51s6/X/Xb6vNXVO6P6rvrx5jLb95SGcqzl41ksdl1eCSv/gayfscQ""",
    "lab:logs/fs01-auth.log": """
eNqdkVtPGzEQhd/5FfNIHog93ksSozxsC1Jf0guhKm1VoV2vl1hkbWN7I8KvrzekZBVRCTGyrLE1Oufz8aJ0gAnQGcecswwaTxG8
X9W/EafTPxwKIaQNsgbbVWsl7uUWGuPAWD8u61ZpaJxpAek4rizuDKxxATI6i20UYhwuL1iW4QyWnwqW5fxh2uLNI12XFfVbHVYy
KEGjLm2UvpPOOqUDLWKdLHq4FGjCGeXpEC6hKeL/6fxG3FaluO/sEI9Fwme6FCd5vqe7Whb/yH5VkwPRjm8jnmV2fB9e6ogsf4XM
lu1tp9XjaX/LvfReGT3isO9igFJH7p6283KIfNqpeo5IJyOotrA70dHAMUk4TfaOXW2iY0ZZdBw8msP19c+5DZ4gnMPXHxdzsjKt
JIORc/i+vLyaO2NC7D9+WSyKz3Gs845USpMQ7c7EUwNkUzoSWkvGDzgOd09AvNuQ2og+yHJN4qeVWkjCKMvPvuEbOA/JxNs3JNMj
7lPoA3ktqKFrytn0Ha5ibfyx60A2Q87wHd98LHugP/kLLqYZ1g==""",
    "lab:logs/firewall.log": """
eNqt291qXDcQB/D7PMW+wC6ab+lACsXpdU1JH8AhDgkYu2Q3hLx9tc42tY4d/7XxgMlV7N9KGs3M0VlxYd8W2ZJsyJcSSymbD18L
bX6/uPjj8u3m7cXlhsqu/2jZWSzaWuXN9rcNNd6VHe+4LKqy+fzl5vr11c3N3dftx8Phn/327sth8+7b4Xr/Ws21veIBovZzyPq/
vFgpOkAEIaJmvoK0QKi1Jg8hmYBUaQWZ4qlz8gFyBEVVlzVUn4WcFjOqA8QQEtE6OLGwIEdFw4Ylqsjh2lqsoYBLVCvTMHMEoRJq
K6jPNxqRxhjdOBZCi60hZTSivo1iGJFCiIN1cOpC2PEqet52leiLtIYUxlxtZYgFVjigHttryGAsKEkbgpscQ/3DDVBbJOCIgric
F3SVYpUXWt8kMLqjthimLuAa0ZhR+6oSTN3dqXUYUMPbtVpZQYx3UfgI4SVS6yl1BU3sIqZVLMCZi2J1HBF15Qf05q8/L78zre6M
OlV20WPBezq5h+517h9gEantu/T++sPVl5vD9v317bcT448MftbwHvpk9kID79LWdFwZgStDEraGHFdv8jGRMs471WQFPZfgTkXV
Y6xBgvNOUaprKJ5dnurHjH2qQb+6PAyMvjzMp3j+VUNgX9XLlQ9pjWEWUG1jWuuQEgwBqirn5U/XXujXkMF0Y1zKeSFgscqffe4J
1+waet7mYVFex7TBzaPVx1oqBjvSvneG1pcZFe1jRSh9jc6D3IJtBSmGqseYqHHpIeayhhoekfGQ4NjwGgmtoWMw8COIi/S/SSS7
HpTauNTVXmV+CO33H7fvrvaHT3e3P3IpPyGRQok4RWLBkqdIgiXWFKnXMyRJSZB6JcJjkkiRKKCkliJxw1JLkQSvk3GKZISljCjv
jT2WXFMkMizVFInxOnlLkRTPXuSMyQqWMsakM1WjRoo0kSOap0gC10mKpEgKZ0+IUiSbkDJmzyaqhjClSDhHSEol7L+lWIoUSRxK
KimSNixljMknqoZoTZEIx57ljInxOllLkXDVEM+ZPcOzFxnVPfB5Wpif98herY4H+ico8JA8ZUiMlyml5MbEo4ZUSZEUp72U4n58
XoZSy3gAqAsVLNUUiWF50pQ2oi4Co1yppEj44VNTGpY20VgqtRQJFw1lTpEYr1NKw9ImGksVTZFw0VB5cYaV+1cWOPa0pEj42Ehf
3oTdSzIxpkiRFMeeZcweTbR7ajVFItiEqVOKhA8JNHJmbyIiImf28COh1oydyxNdmDZKkRiPqbUUSWB1t6IpEj7MMcqYPZmoufby
g/l7iSakSJFwzTXWFEkalixFMtiXm2SMSfsPllqKhHO5pVRCXQTvJ/MUSXGOSKlP2hsxLGXEnk10y+Y1RcK53MJTJHzgaxEpkuLZ
q5wiTeSImrFzvWNQapIiMd65raZI+IzFC6dIuC//77sRL5UMZlinjKoRHcNSpEj4pZCn1NyYeHHsIikSfgLwlCfqmOgsPeWJ+skv
cD+SIkfC66QtRSK8TkYpEu5h3SxFwi8b3HLWCZ8becopwcxJoltLkXBv5K4pEn4h6Sn9Xpt4IekhKdJENqqUI+EckdEbUZnJRs1T
JJyNokiKJHA/RcYpwfH6k2MpY0wzZ5bBJUWiCSlSJPxGMjJOCbqEz8JCOUXC759CM8Y08+XHUEuR8FlYZPQRNHM6GiYpEq4aYTmz
N5EjMqo7yUQPG15SJPwF6XBPkfCbrghNkSZyRM2IPYVPn1aIM7JReyrDPvibSsMqSTne1noMfbo9XH++vbr5cbGhri6GycM7vT+7
S8Xt0RvJcy7SyBP3W/5+czl8dJPhPXhXjk38g9G8v92vR9P4CYZ/MmvfGWl2Oob4f4SMb21VJmllPXP2/My10v9HvOgKktx/5xHc
b7F66ienb4OICbdX/wLJa96W""",
    "lab:logs/docportal-app.jsonl": """
eNrF1kFPgzAUAOC7v2LhLOW9thTGzYuJlxmNiYmXhXXdRtYBgYIa43+3nYlRs0nNogQO9LWkHy+PBy+BaYNsElCgIgQWIrtDllGa
UfYQnE8CrXql3YKr2eW1C9hxaVxgWcluZ69JX6hHN9O1qnETuZSGSK2aLaCLt42cF7WbQSD25EAEcxN1bjYuHK2KMi+livaIG4yK
sg9RJCmplyu3UFbdfk98PXs56I3ZqV6t8iXQI9w4GeamTHhyeUbFCVy7O9mtm5Nyy6fMH5v8JRaHsQjgjeXiTwvBJ7cM+D9yf37P
fApXpL61EGdARy5cjP2xTIxcuCn1xYoMcWQsx6k/duz+BZR7YxmM3WxT9MdOR+9fIhXeXD72t4EL39wmGcDYWOb1R8PvwP7O8IwN
YfPObIiu1kX5Wdr2cr7I5barv1FpQsAee+hOmU21dGFdyVyHptqq0g8W2171FXZ/cTs7nMVFp7dz9VRXjfk18VguPys5UjtaPBvl
pDwFEIj0BzqNB3L6jiWy2tVaGXVc/cFrmz6yD+xuy/UA9vXsDceHGjg=""",
    "br:dia-57/proxy-sa.log": """
eNq9Wk1v20YQvedX8A9I3vnYLwI+BUFu8SW3ogdWUlOhsqRIFODm13e4ayukIHI3rLeADjYEcx7fzrx5M+unVVtVugJVq+5THU+H
l38W56Y6f79s178hAvxey9dLVEtWSw3V6mW52uzb00Hp6uPTly+fPn6tnptjc15uXprn425TM1OF8qzDpX1EQ7ra7h+ZvaVqfW4f
weNSHrcE+PD0M7iRTzI4Vt82p7fokBGdHLsQ3VjtBtHJ96J3H0xGp96rm2vwrfze7DftYrXe38UA1voOg1UEMMRg+xg4h/5fZ8Aq
DgxYq3k8OkItf5SKrqumXb9Fx2v0Pw+n/Wa1WR9Oi1Vzag+bkVRQ8oNAMeDIDKGYPhRd6zQRtncY9orkeDi1zW6xP7Tb1XaMEaZ4
HuxIDWAg92H4GnlmTmQSIpkBHRI0YIaEIPaQEM9HMp4YgBROAwFhWJio+sFdjXbmaUxkJduQlUSO7FAVqBecXS1JPu/N/7icV839
4GRCcLDK6fEE0FISc/UwtyKMvH1IANZmXJ60rtnPlYYcfaJQlqT8EAT/TASola4VFwNB3gcmLIK7KUrTAwEk8lBSGzQHGEZr44cw
dB+Gq9nOVcssHFKdgQ4y1o91TcHhpT5/kY7Pn75Wf7XtsX542O3/fov+cG5sRYKyi87SDLvivJ6EkzxfguqeeBOeYeZpnDenU9P8
uOwWx+3L/bx0HASKHWAEgoqEAwBaWtuDgV1yFoOBXdcWGDqNQmM5FOgjGd5No6AcJzP7SAyGzJAEcQkcohZzcTTtpdltfzSL58P6
sjvcRyJJGQrkKlnXHIWBbKHJATJSquM9BK2PJgLF2A6FYkCDrzWXpAHRvmr3NA3iITJayP1mliVYyoeeqrW64YP0EIY2JfnQHHWT
IcFHlnTNdPzIsaOyvellYG4g2KK58Tp32AQXTOIw3n/w8y7opmM/QQJLQkBJEkjFhBDhmiZBrKZ7f5PdOcww8tBNH0fqB/fzR568
ssAAw6mEWmo5Dl8UyNXmJYDYHCAjbnNCtjnuA7r2deOrsB/dz+/lWTTINB7OAyfPA8Vx16yLlkf0mc77BBBbQ8kMJWBtMZwNqOla
FTCinGXNhdRJV7WYYAW4Jv3+WQrsVLScrMbWAiE4+7L9I9aqUgkWbMnxFL0ObYykp/uxRoLdLKJd0UrRKrKRyAksLqGvLQVtAoip
afZKNXeDpWxghcTy+LFVCtakakVF9dRjHJcTqSpACAvaYHxttFozDl3wAIWeP7Bm6le0XteN7xgdZv5+bcL5YZRPYvRDFsD1grPK
cVz/JSkwrpsdTLPApSsWILYTTkiHFkbs/+G+NCSBIBS+DSAfs4RRRsYxa4zd7rNsrbDFOEFPckLd9RhQWY9uMopWgFDOTmHkcKbu
A+LtDCp7M7D1bmeoWwND2WLhoF0WzDQN3ZVZ0WIhgpgYifMAXSOVlY9ggJzV00BQqrasoJooqDpRKgg12cJXZz6uWLS6VY/etWZ3
WTD/9irTeKg4WGOCEpdDyX3jkb7Bce7NItvhDTP4HobO/JS1Ha9TrfPTZIj/oaLzG8aGa1FNAxELwkX3Lhgv3xNenTItyIhXzzGm
IGNDUHdP1oztq6lbfGQQMnMtSs6EVist1w3/GQP9h38BruHAMg==""",
    "br:dia-57/fw-serraazul.log": """
eNqtmdtqHEcQhu/9FPsCu9S5uwYcCCL3vvALyKAQgxKHSEb47V2zjhJ1zbKlaQv2Slrpmzr9dRgCsiPCEfQAbQGIz+H3p+PD7eHX
m5vfPnw8fLz5cEA4EZwEToqLKroejr8c0OkUPz8hLiJ8+Ofr/d372/v7L0/HPx4f/344fvn6ePj07fHu4b00g/6ORpLF5xqJFhUw
f0lir0gKBpZICAvSNRIv2tlsILWK1AgcMkkK79Ei3tD3kRRFs/cofM7XSLqoSzzMS5JVJOPwQybFf7pqU1tJNNhEUnqvk3km+UJS
xIm740CiikTNuSUSS00K5402QUmi+EIm9YVa4T1mkKGeuCJxY8REkr4AFjaZYtsXJwKRHKfQAKk0oiPyvsolB8/eiywWLzQC0Ics
lzJOJj3lXrhOF5CCZM6DRlBZT82wtUSK6JIWGdFgrNyowcomR7RM6ou0QiO4oe3T8mYdKJM8Er2wCQH7D1KPDDkhrL/EpUOGvWT5
BZJgSSJaSQQcNkUinlqpseJukFihsaAli3EvqwnjFqVUo2w3itrGKi664RklsNuD0aU2iRGJCyUMn5P9v8TAuojRXTLMCljkO7kM
Wku1C5WkZ5JHf6jNogmzumbBiGZ1XQQjdwy1D3JbCkYodGpWZ5JaaRbifrOIeo4WV5Uc/cpBRnWqFZeUL4BaaRVN5CCB5uqSeGq9
Hixx9Z1WIUrLkiERLKzN0hmzSDMsOmSvgkVOQ2mVY4yRWE4L8XrcDLN8wizeyK6GD72EMc3AMIvGeVAuR4w2+BDL8ZYjxFkx1F/R
TZCnfAhjeXmMTYuUXTLyd0Y0iDJs3eYqGLcJGGN0L4TMC+2oPakyIfSNNcFie2SttkcT2rf/mBFsQVKnvbaZrpzm3YC1coaPnRh0
nyIKcOuZFGlfTqFoOCX0nmD0Kul4PirsS3vMaR+TDVe3Eupt5w7OrJLNYliAa7P8Tao5YEzFZMO9DQs/l6AOlnOQ8eJk+P946+v8
3exHHZ/xemHhf7r79Oft5/vj57+eaws3tRUo1AoVje9tUJf28AElTd8ERRdXoeRA+nc0fD2qd9kke8yu9dqFbUqdNGsuW3016a0P
2/i6VxeVZeJZneLxoFanPtFKom1hhr1KnRxn+pZlH2pYVg7YNLPkEUvfwgiLJS9WNt43icYcuun9+po8jAqYCtgwReF6dF+Qa9jU
yNYhw7hYX3WxGIZk3/GYYiXfkDS6cmkW8s8vyuuRPz51HqLPCMeY9CtJF6oDRvzzl42AhZJSqR3EM5MNjofkFYYLt+o43o13vsQg
aTlgpPVxPMyaGjbGy+EK64VZeL7ZDGZhebVmZ8tJv441tXDIzGgIZhmmxQx1humM0jtn2PpHUMNmKqxzzsOyh60vCJVo3zmPQTmX
l3ph1vrqBNpwYSOvTTKFd98BEId8CA==""",
    "br:dia-57/sa-cx-07-security.csv": """
eNqdklFLwzAUhd/9FaNPCulI0naWvI1twsCNYTtEyUtorzXYNiXJXPXXm07RdTKdQh7CzZecc25uKiuYaBAW8rXN0OwZajufolTo
AuzagF6KCtC1KlSdvjSA5s04zzUYg5awXWmVue0OWQntru5VziimI59gH0cpwSwiDNN7FI5oiJK7JJ0tUIR85HluHaKUEfyBZu0w
c+9qhS8RPc4HpOPjuMcj5E0Yv5V1rraGQ9uUSoMeQgte7yR5MRaqgPKNyytraXfIoUjAQnxUxOUutKgGV7IEMzhv49EFX8hMK6Me
LJ/lBfBx05QyE1aqmlcGXOm7k57HngHCaPhTyu6rDN+rd3pTYQW/UaKSdcEXydjpCl/YjSjlq8iV/rOBMPy1zZ/NNNmjFebJfIn8
32PPScBIxMj7LEUnDMiOp6cPlPtoynDHxxgfJt3RbzPg96g=""",
    "br:dia-57/sa-cx-07-mft.csv": """
eNqNkM1qhDAUhffzFPMAMdwbM/7tpNPuZhgq3QnlVlMb0ESiUuvT159FocgwIYsDB77zJc+mdz/XoflQjt3IKdPfqP9iL7pWV2rU
GjI9KfbkFPWqhBHhL/twQAGxj4znb51yXV6MvJghzkKYn+23qS2VHbvYcqjte6aqwZEpaE7OUToNNZ90yyRCICQTIAIPwYPTETER
foKSC4Er/063Kki5q5C27Zl6yl8tNdpU7JKlDP7DZCIiHgPOPLzXbUOnR4byZajpyKN+oFpPVFrH1aiYH8UzZVlBD4QH/hEhgeVy
2M6OQczl/EUQhZtB8LBBYc2nrrg2moVS7pIDmB/n3+0Ov7/omTQ=""",
    "br:dia-57/webmail-sa.jsonl": """
eNrFk00LwjAMhu/+itGzC0m7KvZ3ePIis0QdzE3aVQbif3cdeFD8RMaub0KevPk4i8YLkwiJcpYSpqiXhEYrg3IlpongE1dNTMhD
s4ey3hVVlINnF1Xbgu0SXI3zKHtn18UxBghBImQIug849qHs6/hgLXsvLpPzc3RmMvoGvWN3Y9MrtvyFLY2if21LVIBApGBB9+xt
XpTB8Ts26fHYGY7FlgaH8v1x38roB9+Hrt1N3YILJa+t47zhQZvoHk0bHOrg3w+/Z9Ni8Ge7Aj6RNWk=""",
}

REGISTRO_DE_CARGA = []   # (arquivo, origem, sha256): vai para as suas notas de método


def carregar(chave):
    """chave = 'lab:logs/firewall.log' ou 'br:dia-57/proxy-sa.log'. Devolve o texto."""
    prefixo, rel = chave.split(":", 1)                    # separa de onde vem e o caminho
    url = (RAW_LAB if prefixo == "lab" else RAW_BR) + rel  # monta a URL raw
    reserva = zlib.decompress(base64.b64decode(EMBUTIDOS[chave].replace("\n", "")))  # bytes da cópia embutida
    try:
        with urllib.request.urlopen(url, timeout=10) as r:  # tenta a rede, com tempo limite
            dados, origem = r.read(), "rede"
    except Exception:                                       # sem rede, 404, timeout: tanto faz
        dados, origem = reserva, "embutido"
    if origem == "rede" and dados != reserva:              # a versão online mudou desde que copiamos?
        print(f"⚠️  {rel}: versão online difere da embutida; usando a embutida (números do dia batem com ela)")
        dados, origem = reserva, "embutido (online divergente)"
    sha = hashlib.sha256(dados).hexdigest()                # impressão digital do que vamos analisar
    REGISTRO_DE_CARGA.append((rel, origem, sha))           # anota para o relatório
    return dados.decode("utf-8")


print(f"{len(EMBUTIDOS)} arquivos disponíveis. Use carregar('...').")

In [ ]:
# O pacote EVID-001 do LAB-P4: quatro arquivos, quatro formatos.
EVID001 = {
    "bastion01-auth.log":  carregar("lab:logs/bastion01-auth.log"),
    "fs01-auth.log":       carregar("lab:logs/fs01-auth.log"),
    "firewall.log":        carregar("lab:logs/firewall.log"),
    "docportal-app.jsonl": carregar("lab:logs/docportal-app.jsonl"),
}
for nome, texto in EVID001.items():                 # quantas linhas tem cada um
    print(f"{nome:22s} {len(texto.splitlines()):4d} linhas")

In [ ]:
# Tabela de carga: o que foi lido, de onde e com qual hash.
for rel, origem, sha in REGISTRO_DE_CARGA:     # uma linha por arquivo
    print(f"{rel:40s} {origem:10s} {sha[:16]}…")  # 16 primeiros caracteres bastam para conferir de olho

## 2. Um mini-Plaso: parsers, filestat e o silêncio

O Plaso tem centenas de *parsers*. O nosso tem **um**: syslog (o formato `Mmm dd HH:MM:SS host prog[pid]: msg`). Para todo arquivo, além disso, ele gera um evento `filestat` com a data de modificação do arquivo, como o Plaso real.

Repare no que acontece com `firewall.log` e `docportal-app.jsonl`: **nenhum erro**. Só zero eventos de conteúdo.

In [ ]:
import re                                   # expressões regulares para reconhecer o formato syslog
from datetime import datetime, timezone     # datas com fuso

# Formato syslog clássico: "Mar 14 03:14:07 bastion01 sshd[21927]: Accepted ..."
# Note que NÃO há ano. Guarde isso.
SYSLOG = re.compile(r"^(\w{3})\s+(\d{1,2}) (\d\d:\d\d:\d\d) (\S+) ([^:\[]+)(?:\[\d+\])?: (.*)$")
MESES = {m: i for i, m in enumerate(["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"], 1)}


def inferir_ano(mes_evento, mtime):
    """
    Heurística parecida com a do Plaso: o ano é o da data de modificação do arquivo;
    se o mês do evento é DEPOIS do mês do mtime, o evento deve ser do ano anterior.
    """
    return mtime.year - 1 if mes_evento > mtime.month else mtime.year


def parser_syslog(nome, texto, mtime):
    """Devolve eventos (dicts) ou None se o arquivo não parece syslog."""
    linhas = texto.splitlines()
    if not linhas or not SYSLOG.match(linhas[0]):    # o Plaso também "fareja" o formato antes
        return None
    eventos = []
    for n, linha in enumerate(linhas, start=1):
        m = SYSLOG.match(linha)
        if not m:
            continue                                  # linha estranha: pula (o Plaso gera um aviso)
        mes, dia, hora, host, prog, msg = m.groups()
        ano = inferir_ano(MESES[mes], mtime)          # 👈 o ano é ADIVINHADO
        dt = datetime.strptime(f"{ano}-{MESES[mes]:02d}-{int(dia):02d} {hora}", "%Y-%m-%d %H:%M:%S")
        eventos.append({"dt": dt.replace(tzinfo=timezone.utc), "parser": "syslog", "host": host,
                        "fonte": nome, "linha": n, "desc": f"{prog}: {msg}"})
    return eventos


PARSERS = [parser_syslog]                             # o nosso "log2timeline --info": só isto


def log2timeline(pacote, mtime):
    """Roda todos os parsers em todos os arquivos. Nunca reclama de formato desconhecido."""
    armazenamento, cobertura = [], {}
    for nome, texto in pacote.items():
        # filestat: todo arquivo gera o evento da sua data de modificação (a hora da CÓPIA!)
        armazenamento.append({"dt": mtime, "parser": "filestat", "host": "-", "fonte": nome,
                              "linha": 0, "desc": "File Modification Time"})
        cobertura[nome] = {"filestat": 1}
        for p in PARSERS:
            evs = p(nome, texto, mtime)
            if evs:                                   # só registra se o parser reconheceu o arquivo
                armazenamento.extend(evs)
                cobertura[nome][p.__name__.replace("parser_", "")] = len(evs)
        # 👇 repare: não há 'else: print("não entendi este arquivo")'. É o silêncio do Plaso.
    return sorted(armazenamento, key=lambda e: e["dt"]), cobertura


# A cópia de trabalho foi feita em 2026-03-20 (data de modificação dos arquivos copiados)
MTIME_COPIA = datetime(2026, 3, 20, 9, 0, tzinfo=timezone.utc)
plaso_lab, cobertura_lab = log2timeline(EVID001, MTIME_COPIA)
print(f"lab-p4-logs.plaso (simulado): {len(plaso_lab)} eventos")

In [ ]:
# O nosso "pinfo": eventos por parser, por arquivo.
def pinfo(cobertura):
    print(f"{'Arquivo':22s} {'Parsers':28s} Eventos")
    for nome, por_parser in cobertura.items():
        parsers = ", ".join(f"{p}={n}" for p, n in por_parser.items())
        total = sum(por_parser.values())
        alerta = "  ⚠️ só filestat: CONTEÚDO NÃO INTERPRETADO" if list(por_parser) == ["filestat"] else ""
        print(f"{nome:22s} {parsers:28s} {total:5d}{alerta}")

pinfo(cobertura_lab)

## 3. Tabela de cobertura (vai para o relatório do Dia 64)

A coluna **Problema** é sua. A ferramenta não preenche.

In [ ]:
PROBLEMAS_LAB = {                                             # o que você sabe dos dias 55 e 56
    "bastion01-auth.log":  "Ano inferido",
    "fs01-auth.log":       "Relógio 83 s adiantado, sem correção",
    "firewall.log":        "Formato próprio; hora local (America/New_York)",
    "docportal-app.jsonl": "Formato JSON próprio; mesmo desvio de 83 s do fs01",
}

def tabela_cobertura(cobertura, problemas):
    linhas = ["| Arquivo de origem | Parser que o tratou | Contagem de eventos | Problema |",
              "|---|---|---|---|"]
    for nome, por_parser in cobertura.items():
        conteudo = [p for p in por_parser if p != "filestat"]           # parsers de conteúdo
        parser = ", ".join(conteudo) if conteudo else "nenhum (só filestat)"
        linhas.append(f"| {nome} | {parser} | {sum(por_parser.values())} | {problemas.get(nome, '?')} |")
    return "\n".join(linhas)

print(tabela_cobertura(cobertura_lab, PROBLEMAS_LAB))

## 4. O ano adivinhado: quando ele erra

Mude `ano_da_copia` no formulário e rode. Se a cópia de trabalho tivesse sido feita em maio de 2027 (por exemplo, um caso reaberto), qual ano o syslog ganharia? Depois teste janeiro de 2027 (`mes_da_copia = 1`): por que aí a heurística acerta?

In [ ]:
#@title Simular a cópia em outra data { display-mode: "form" }
ano_da_copia = 2027  #@param {type:"integer"}
mes_da_copia = 5  #@param {type:"integer"}

mtime_errado = datetime(ano_da_copia, mes_da_copia, 15, tzinfo=timezone.utc)
evs, _ = log2timeline({"bastion01-auth.log": EVID001["bastion01-auth.log"]}, mtime_errado)
aceitos = [e for e in evs if "Accepted password" in e["desc"]]   # o login que importa
for e in aceitos[:3]:
    print(e["dt"].isoformat(), e["desc"][:60])
print("\n➡️  Ano certo: 2026. Se apareceu outro, é o caso de declarar o ano (--preferred_year 2026) e registrar por quê.")

## 5. psort: a janela do incidente e a fatia de ±5 min

Formato `l2tcsv`, as 17 colunas clássicas. Filtramos `02:45 < data < 04:00` UTC de 14/03/2026.

In [ ]:
import csv, io                                    # escrever CSV sem dor
from datetime import timedelta

COLUNAS_L2T = ["date","time","timezone","MACB","source","sourcetype","type","user","host",
               "short","desc","version","filename","inode","notes","format","extra"]

def psort_l2tcsv(eventos, inicio, fim, caminho):
    """Grava só os eventos estritamente dentro da janela (como o filtro 'date > ... AND date < ...')."""
    janela = [e for e in eventos if inicio < e["dt"] < fim]
    with open(caminho, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(COLUNAS_L2T)
        for e in janela:
            w.writerow([e["dt"].strftime("%m/%d/%Y"), e["dt"].strftime("%H:%M:%S"), "UTC", "....",
                        "LOG", e["parser"], "Content Modification Time" if e["parser"] == "filestat" else "Entry Written",
                        "-", e["host"], e["desc"][:60], e["desc"], "2", e["fonte"], "-", "-",
                        e["parser"], f"linha={e['linha']}"])
    return janela

INICIO = datetime(2026, 3, 14, 2, 45, tzinfo=timezone.utc)
FIM = datetime(2026, 3, 14, 4, 0, tzinfo=timezone.utc)
janela = psort_l2tcsv(plaso_lab, INICIO, FIM, "lab-p4-window.csv")
print(f"lab-p4-window.csv: {len(janela)} eventos, de {janela[0]['dt']:%H:%M:%S} a {janela[-1]['dt']:%H:%M:%S} UTC")

# --slice: 5 minutos para cada lado do primeiro login bem-sucedido
centro = datetime(2026, 3, 14, 3, 14, 7, tzinfo=timezone.utc)
fatia = [e for e in plaso_lab if abs(e["dt"] - centro) <= timedelta(minutes=5)]
print(f"fatia ±5 min em torno de 03:14:07: {len(fatia)} eventos")
for e in fatia[-4:]:
    print("  ", e["dt"].strftime("%H:%M:%S"), e["host"], e["desc"][:70])

## 6. Plaso × sua linha do tempo do Dia 56

O que cada uma tem que a outra não tem? E por que o `fs01` aparece 83 s depois aqui?

In [ ]:
fs01 = [e for e in janela if e["host"] == "fs01"]               # eventos do fs01 na janela do Plaso
for e in fs01[:2]:
    corrigido = e["dt"] - timedelta(seconds=83)                  # a correção medida no Dia 56
    print(f"Plaso: {e['dt']:%H:%M:%S}  |  Dia 56 (corrigido): {corrigido:%H:%M:%S}  |  {e['desc'][:50]}")

print("""
Plaso tem e o script não tinha : eventos filestat (tempos de arquivo da cópia).
Script tem e o Plaso não tem   : TODO o conteúdo de firewall.log e docportal-app.jsonl,
                                 e a correção de -83 s do fs01.
No relatório: a hora CORRIGIDA, com a medição que a sustenta (porta 41766).""")

## 7. ➕ O pacote do `SA-CX-07` (Operação Isca)

Agora o mesmo exercício no caso brasileiro. **Antes de rodar**, preveja: quais dos cinco arquivos o nosso mini-Plaso vai interpretar?

In [ ]:
SA = {
    "proxy-sa.log":          carregar("br:dia-57/proxy-sa.log"),
    "fw-serraazul.log":      carregar("br:dia-57/fw-serraazul.log"),
    "sa-cx-07-security.csv": carregar("br:dia-57/sa-cx-07-security.csv"),
    "sa-cx-07-mft.csv":      carregar("br:dia-57/sa-cx-07-mft.csv"),
    "webmail-sa.jsonl":      carregar("br:dia-57/webmail-sa.jsonl"),
}
MTIME_SA = datetime(2026, 10, 6, 14, 0, tzinfo=timezone.utc)   # a TI copiou os arquivos em 6/out
plaso_sa, cobertura_sa = log2timeline(SA, MTIME_SA)
pinfo(cobertura_sa)

In [ ]:
# O firewall da cooperativa grava hora de Brasília SEM fuso. A correção é nossa, fora do "Plaso".
try:
    from zoneinfo import ZoneInfo                    # stdlib desde o Python 3.9
    SP = ZoneInfo("America/Sao_Paulo")               # banco IANA, com histórico de horário de verão
except Exception:                                    # Windows sem o pacote tzdata, por exemplo
    SP = timezone(timedelta(hours=-3), "BRT")        # plano B: UTC-3 fixo (válido em 2026)

def fw_para_utc(linha):
    data, hora = linha.split()[:2]                                     # "2026-10-05 08:19:38"
    local = datetime.fromisoformat(f"{data} {hora}").replace(tzinfo=SP)  # "isto estava no relógio de SP"
    return local.astimezone(timezone.utc)

clique_fw = next(l for l in SA["fw-serraazul.log"].splitlines() if "198.51.100.201" in l)
clique_proxy = next(e for e in plaso_sa if "lnk.example/sa7" in e["desc"])
print("firewall (local):", clique_fw[:19])
print("firewall (UTC)  :", fw_para_utc(clique_fw).isoformat())
print("proxy (UTC)     :", clique_proxy["dt"].isoformat(), "← mesmo segundo: é o clique da caixa")

In [ ]:
# A armadilha do "-z America/Sao_Paulo": o Plaso aplica UM fuso à execução inteira.
# Se aplicássemos UTC-3 ao proxy (que já está em UTC), o clique "andaria" 3 horas:
errado = clique_proxy["dt"].replace(tzinfo=SP).astimezone(timezone.utc)
print("proxy com -z America/Sao_Paulo:", errado.isoformat(), "❌ (3 h depois do real)")
print("E o firewall continuaria fora: o conteúdo dele nem é interpretado.")

PROBLEMAS_SA = {
    "proxy-sa.log":          "Syslog sem ano (inferido); relógio em UTC",
    "fw-serraazul.log":      "Formato próprio; hora de Brasília sem fuso",
    "sa-cx-07-security.csv": "CSV exportado, não .evtx: sem parser",
    "sa-cx-07-mft.csv":      "CSV exportado da MFT: sem parser (o Plaso leria a MFT na imagem)",
    "webmail-sa.jsonl":      "JSON próprio",
}
print()
print(tabela_cobertura(cobertura_sa, PROBLEMAS_SA))

## 8. 🎮 Quiz da cobertura

Seis perguntas. Responda `a`, `b` ou `c`. Checkpoint BR: pelo menos 5.

In [ ]:
# ------------------------------------------------------------
# QUIZ: função genérica de múltipla escolha
# MODO_JOGO = True  -> pergunta e conta acertos
# MODO_JOGO = False -> só mostra gabarito e explicação
# ------------------------------------------------------------
def jogar_quiz(perguntas, modo_jogo=True, minimo=None):
    acertos = 0                                            # placar
    for n, (enunciado, opcoes, certa, porque) in enumerate(perguntas, start=1):
        print(f"\n[{n}/{len(perguntas)}] {enunciado}")    # o enunciado
        for letra, texto in opcoes.items():                # as alternativas a), b), c)
            print(f"   {letra}) {texto}")
        if modo_jogo:
            resposta = ""
            for _ in range(5):                             # até 5 tentativas de digitar uma letra válida
                resposta = input("   Sua resposta: ").strip().lower()
                if resposta in opcoes:
                    break
            ok = (resposta == certa)                       # acertou?
            acertos += ok
            print("   ✅ Acertou!" if ok else f"   ❌ Era {certa}).")
        else:
            print(f"   Gabarito: {certa})")
        print(f"   Por quê: {porque}")
    if modo_jogo:
        print(f"\n🏁 Resultado: {acertos}/{len(perguntas)}")
        if minimo is not None:
            print("Checkpoint BR cumprido. 🥷" if acertos >= minimo
                  else f"Checkpoint BR pede pelo menos {minimo}. Releia o README e tente de novo.")
    return acertos

In [ ]:
MODO_JOGO = True   # False = só mostra o gabarito

PERGUNTAS = [
    ("O pinfo mostra 'filestat=1' e nada mais para firewall.log. Isso significa:",
     {"a": "que o firewall não registrou nada na noite do incidente",
      "b": "que o conteúdo do arquivo não foi interpretado; só a data do arquivo entrou",
      "c": "que o Plaso deu erro e precisa ser reinstalado"},
     "b", "O Plaso pula formatos sem parser em silêncio. O filestat é só a hora da sua cópia."),
    ("O syslog do proxy-sa não tem ano. A cópia foi feita em 6/out/2026. O ano inferido é:",
     {"a": "2026", "b": "2025", "c": "impossível saber"},
     "a", "Mês do evento (out) não é posterior ao mês do mtime (out), então fica o ano do mtime."),
    ("Rodar o Plaso com -z America/Sao_Paulo sobre o pacote do SA-CX-07 faria:",
     {"a": "corrigir o firewall e o proxy de uma vez",
      "b": "deslocar 3 h os eventos do proxy, que já estão em UTC, sem tocar no firewall",
      "c": "nada, porque o Plaso ignora fusos"},
     "b", "Um fuso por execução; o proxy é UTC e o firewall nem é interpretado."),
    ("A linha do firewall '2026-10-05 08:19:38' em UTC é:",
     {"a": "05:19:38Z", "b": "08:19:38Z", "c": "11:19:38Z"},
     "c", "Brasília é UTC-3: some 3 horas. Bate com o GET em lnk.example/sa7 no proxy."),
    ("No relatório, para eventos do fs01 você usa:",
     {"a": "a hora do Plaso (83 s adiantada)",
      "b": "a hora corrigida, com a medição que sustenta a correção",
      "c": "a média das duas"},
     "b", "A correção vale porque foi medida (porta 41766) e documentada."),
    ("Por que a tabela de cobertura vai para a seção de método do relatório?",
     {"a": "porque o leitor precisa saber o que a super-linha do tempo NÃO mostra",
      "b": "para deixar o relatório mais longo",
      "c": "porque o Plaso exige"},
     "a", "É a limitação declarada antes que a defesa a encontre."),
]
jogar_quiz(PERGUNTAS, MODO_JOGO, minimo=5)

## 9. (Opcional) O Plaso de verdade

Só funciona onde houver Docker (sua VM de laboratório; o Colab não tem Docker). As linhas com `!` são puladas pelo teste do repositório.

In [ ]:
# Rode na sua VM de laboratório, não no Colab.
!docker run --rm log2timeline/plaso log2timeline --version
!docker run --rm -v ~/lab-p4:/data log2timeline/plaso pinfo /data/work/lab-p4-logs.plaso
print("Se você está no Colab, as linhas acima não rodam (não há Docker). Tudo bem: a lógica você já viu.")

## 10. 💾 Levando os artefatos

In [ ]:
from pathlib import Path
Path("notas").mkdir(exist_ok=True)
Path("notas/cobertura-lab-p4.md").write_text(tabela_cobertura(cobertura_lab, PROBLEMAS_LAB) + "\n", encoding="utf-8")
Path("notas/cobertura-sa-cx-07.md").write_text(tabela_cobertura(cobertura_sa, PROBLEMAS_SA) + "\n", encoding="utf-8")
print("💾 notas/cobertura-lab-p4.md, notas/cobertura-sa-cx-07.md e lab-p4-window.csv")
try:
    from google.colab import files                     # só existe no Colab
    for f in ("lab-p4-window.csv", "notas/cobertura-lab-p4.md", "notas/cobertura-sa-cx-07.md"):
        files.download(f)
except ImportError:
    print("(Fora do Colab: os arquivos ficaram na pasta atual.)")

## ✅ Antes de fechar o Dia 57

- [ ] Tabela de cobertura do LAB-P4 com a coluna Problema preenchida
- [ ] `lab-p4-window.csv` entre 02:45 e 04:00 UTC
- [ ] Ano confirmado (2026) e o método anotado
- [ ] Tabela de cobertura do `SA-CX-07`, com o firewall convertido para UTC fora do Plaso
- [ ] Quiz com pelo menos 5/6

**Amanhã (Dia 58):** Timesketch. Você vai juntar o que o Plaso leu com o que ele não leu e escrever a história da invasão, evento por evento.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>